# ==============================================================================
# 🌌 ASTROPHYSICAL DISCOVERY: PULSAR STAR DETECTION VIA SUPPORT VECTOR CLASSIFIER
# ==============================================================================
### Core Principles:
Pulsars are highly magnetized rotating neutron stars emitting beams of electromagnetic radiation.
In large-scale radio telescope sky surveys, pulsar signals are exceedingly rare (~9%), drowned out by radio frequency interference (RFI).

Support Vector Classifiers solve extreme class imbalance via cost-sensitive margin weights ($C_+ > C_-$):
$$\min_{\mathbf{w}, b} \frac{1}{2}\|\mathbf{w}\|^2 + C_+ \sum_{i \in +} \xi_i + C_- \sum_{j \in -} \xi_j$$


In [1]:
# STEP 1: ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Astrophysics Environment ready.")


✅ STEP 1: Astrophysics Environment ready.


## 📥 STEP 2 & 3: INGESTION & DATASET HARMONIZATION
Loading Kaggle Pulsar Star HTRU2 dataset (12,528 candidate radio signals).


In [2]:
# STEP 2 & 3: LOAD & HARMONIZE
data_path = 'data/pulsar_star/pulsar_data_train.csv'
df = pd.read_csv(data_path)

df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
print(f"📊 Dataset Shape: {df.shape[0]} candidates, {df.shape[1]} astronomical features")
print(f"Class Distribution:\n{df['target_class'].value_counts(normalize=True).round(4) * 100}%")
df.head(3)


📊 Dataset Shape: 12528 candidates, 9 astronomical features
Class Distribution:
target_class
0.0    90.8
1.0     9.2
Name: proportion, dtype: float64%


,mean_of_the_integrated_profile,standard_deviation_of_the_integrated_profile,excess_kurtosis_of_the_integrated_profile,skewness_of_the_integrated_profile,mean_of_the_dm-snr_curve,standard_deviation_of_the_dm-snr_curve,excess_kurtosis_of_the_dm-snr_curve,skewness_of_the_dm-snr_curve,target_class
0,121.156250,48.372971,0.375485,-0.013165,3.168896,18.399367,7.449874,65.159298,0.0
1,76.968750,36.175557,0.712898,3.388719,2.399666,17.570997,9.414652,102.722975,0.0
2,130.585938,53.229534,0.133408,-0.297242,2.743311,22.362553,8.508364,74.031324,0.0


## 🧹 STEP 4 & 5: SEGREGATION & HANDLING TELESCOPE MISSING VALUES
Imputing sensor dropouts and segregating features from `target_class`.


In [3]:
# STEP 4 & 5: SEGREGATION & HYGIENE
from sklearn.impute import SimpleImputer

X = df.drop(columns=['target_class'])
y = df['target_class'].astype(int)

# Check missing readings
missing_counts = X.isnull().sum()
print(f"Features with missing signals:\n{missing_counts[missing_counts > 0]}")


Features with missing signals:
excess_kurtosis_of_the_integrated_profile    1735
standard_deviation_of_the_dm-snr_curve       1178
skewness_of_the_dm-snr_curve                  625
dtype: int64


## 🔒 STEP 6: STRATIFIED TRAIN-TEST SPLIT
80/20 stratified split to preserve astronomical signal rarity.


In [4]:
# STEP 6: STRATIFIED SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train Signals: {X_train.shape[0]} ({y_train.sum()} Real Pulsars)")
print(f"Test Signals : {X_test.shape[0]} ({y_test.sum()} Real Pulsars)")


Train Signals: 10022 (922 Real Pulsars)
Test Signals : 2506 (231 Real Pulsars)


## ⚖️ STEP 7: COST-SENSITIVE PIPELINE (Standard vs Balanced $C$)
Demonstrating how `class_weight='balanced'` boosts pulsar detection recall.


In [5]:
# STEP 7: CLASS WEIGHT COMPARISON
pipe_unweighted = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('svc', SVC(kernel='rbf', random_state=42))
])
pipe_unweighted.fit(X_train, y_train)

pipe_balanced = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('svc', SVC(kernel='rbf', class_weight='balanced', random_state=42))
])
pipe_balanced.fit(X_train, y_train)

rec_unweighted = recall_score(y_test, pipe_unweighted.predict(X_test))
rec_balanced = recall_score(y_test, pipe_balanced.predict(X_test))

print("="*65)
print(f"Standard SVC Recall       : {rec_unweighted * 100:.2f}%")
print(f"Cost-Sensitive SVC Recall : {rec_balanced * 100:.2f}%")
print(f"Discovery Sensitivity Gain: +{(rec_balanced - rec_unweighted) * 100:.2f}% Pulsars captured!")
print("="*65)


Standard SVC Recall       : 82.25%
Cost-Sensitive SVC Recall : 90.48%
Discovery Sensitivity Gain: +8.23% Pulsars captured!


## 🎯 STEP 8 & 9: HYPERPARAMETER TUNING VIA 3-FOLD CV
Optimizing $C$ and $\gamma$ for harmonic F1-score balance.


In [6]:
# STEP 8 & 9: GRID SEARCH
param_grid = {
    'svc__C': [1, 10, 50],
    'svc__gamma': ['scale', 0.05, 0.1]
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
pipe_tune = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('svc', SVC(kernel='rbf', class_weight='balanced', random_state=42))
])

grid = GridSearchCV(pipe_tune, param_grid=param_grid, cv=cv, scoring='f1', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Astrophysics Hyperparameters: {grid.best_params_}")
print(f"🎯 Best 3-Fold CV F1-Score: {grid.best_score_ * 100:.2f}%")


🏆 Best Astrophysics Hyperparameters: {'svc__C': 50, 'svc__gamma': 0.05}
🎯 Best 3-Fold CV F1-Score: 84.82%


## 📊 STEP 10 & 11: OBSERVATORY EVALUATION & CONFUSION MATRIX
Reviewing precision, recall, and false alarm rejection.


In [7]:
# STEP 10 & 11: EVALUATION REPORT
y_pred = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

print("📋 RADIO OBSERVATORY DETECTION REPORT:")
print(classification_report(y_test, y_pred, target_names=['Noise/RFI', 'Pulsar Star']))

print(f"🎯 Detection F1-Score: {f1_score(y_test, y_pred)*100:.2f}%")
print(f"🎯 Pulsar Recall     : {recall_score(y_test, y_pred)*100:.2f}%")
print(f"🎯 Pulsar Precision  : {precision_score(y_test, y_pred)*100:.2f}%")

print(f"\nConfusion Matrix:\n{cm}")
tn, fp, fn, tp = cm.ravel()
print(f"Confirmed Pulsars Caught : {tp}/{tp+fn}")
print(f"False Radio Noise Alarms : {fp}")


📋 RADIO OBSERVATORY DETECTION REPORT:
              precision    recall  f1-score   support

   Noise/RFI       0.99      0.98      0.98      2275
 Pulsar Star       0.81      0.92      0.86       231

    accuracy                           0.97      2506
   macro avg       0.90      0.95      0.92      2506
weighted avg       0.97      0.97      0.97      2506

🎯 Detection F1-Score: 86.00%
🎯 Pulsar Recall     : 91.77%
🎯 Pulsar Precision  : 80.92%

Confusion Matrix:
[[2225   50]
 [  19  212]]
Confirmed Pulsars Caught : 212/231
False Radio Noise Alarms : 50


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & STREAMING SMOKE TEST
Deploying pipeline and validating observatory telemetry streaming latency.


In [8]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/pulsar_star_svc_pipeline.joblib'
joblib.dump(best_model, model_path, compress=3)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_signal = X_test.iloc[[0]]

t0 = time.perf_counter()
pred_class = loaded_pipe.predict(sample_signal)[0]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n📡 LIVE RADIO TELESCOPE TELEMETRY AUDIT:")
print(f"   Signal Classification: {'✨ CONFIRMED PULSAR STAR' if pred_class == 1 else '📻 NOISE / RFI'}")
print(f"   Inference Latency    : {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/pulsar_star_svc_pipeline.joblib (95,334 bytes)



📡 LIVE RADIO TELESCOPE TELEMETRY AUDIT:
   Signal Classification: 📻 NOISE / RFI
   Inference Latency    : 4.831 ms (SLA < 2.0ms: CHECK)
